# 12 — Deep Hedging multi-instruments : couvrir le vega sous Heston

## Où on en est, et pourquoi cette brique

Jusqu'ici on a couvert un call qu'on vend (le *passif*) en tradant **uniquement le sous-jacent** $S$. Sous Heston le marché est **incomplet** : le prix dépend de deux aléas, le mouvement de $S$ (piloté par $W_1$) *et* le mouvement de la variance $v$ (piloté par $W_2$), et avec le seul sous-jacent on ne peut annuler que le premier. Le risque de variance (le *vega*) reste, et c'est lui qui met un plancher sous le CVaR : ni le delta pur ni le réseau mono-instrument ne descendent en dessous.

Le passif a une sensibilité $\partial P/\partial v$ (vega). Le sous-jacent, lui, a un vega nul : $S$ ne dépend pas de $v$. Donc **aucune quantité de $S$ ne couvre le vega**. Pour l'attraper il faut un instrument qui, lui aussi, dépend de $v$ : une **autre option**. On ajoute donc un call de couverture, de maturité plus longue ($T_2=2$ ans, pour qu'il soit encore vivant à l'échéance $T_1=1$ du passif), qu'on peut acheter/vendre en cours de route.

Le plan de cette brique :
1. **l'idée clé** qui rend tout ça calculable : le marché est *exogène*.
2. le **benchmark classique** delta-vega (couverture par les grecques de Heston).
3. le **réseau multi-instruments** : il sort maintenant un vecteur $(h_S, h_O)$ de deux positions, entraîné sur le CVaR avec coûts.


## 1. L'idée clé : le marché est exogène

Dans ce modèle on ne suppose **aucun impact de marché** : nos trades ne bougent ni $S$ ni $v$. Les trajectoires $(S_t, v_t)$ ne dépendent donc **pas** des paramètres du réseau. Conséquence énorme :

- le prix de l'option de couverture le long d'une trajectoire, $O_t = C_{\text{Heston}}(S_t, v_t, \tau = T_2 - t)$, est une **donnée** fixée d'avance, pas quelque chose à travers quoi il faut backpropager ;
- on peut donc **pré-simuler** un grand paquet de trajectoires $(S, v)$ *une seule fois*, puis **pré-tabuler** le prix de couverture $O$ le long de chacune, et enfin entraîner le réseau sur ces tenseurs figés.

C'est plus simple *et* plus rapide que la boucle fusionnée des notebooks 09/10 : le gradient du CVaR ne circule qu'à travers les **actions** du réseau (combien on achète), jamais à travers le marché. On price l'option Heston par **tabulation sur grille** $(S, v, \tau)$ + interpolation linéaire (grilles calculées dans le script `heston_hedge_grid.npz` et `heston_liab_grid.npz`, à partir du pricer par fonction caractéristique vérifié au notebook 11).


In [ ]:
import numpy as np
import torch
import time
import matplotlib.pyplot as plt
from scipy.integrate import quad
from scipy.interpolate import RegularGridInterpolator

torch.manual_seed(0)
rng = np.random.default_rng(0)

# --- parametres marche (identiques a tout le projet) ---
S0, K, mu, r, T1 = 100., 100., 0.05, 0.02, 1.0     # passif : call K=100, maturite T1=1
v0, kappa, theta, xi, rho = 0.04, 2.0, 0.04, 0.3, -0.7
Kh, T2 = 100., 2.0                                  # option de couverture : call Kh=100, maturite T2=2
n, cost, alpha = 63, 0.01, 0.95                     # 63 pas (~hebdo), cout 1%, CVaR 95%
dt = T1/n

# --- pricer Heston par fonction caracteristique (verifie au notebook 11) ---
def heston_cf(phi, S0, v0, r, kappa, theta, xi, rho, T):
    out = []
    for u, b in [(0.5, kappa - rho*xi), (-0.5, kappa)]:
        d = np.sqrt((rho*xi*1j*phi - b)**2 - xi**2*(2*u*1j*phi - phi**2))
        g = (b - rho*xi*1j*phi + d)/(b - rho*xi*1j*phi - d)
        C = r*1j*phi*T + (kappa*theta/xi**2)*((b - rho*xi*1j*phi + d)*T - 2*np.log((1-g*np.exp(d*T))/(1-g)))
        D = (b - rho*xi*1j*phi + d)/xi**2 * ((1-np.exp(d*T))/(1-g*np.exp(d*T)))
        out.append(np.exp(C + D*v0 + 1j*phi*np.log(S0)))
    return out

def heston_call(S0, v0, r, kappa, theta, xi, rho, T, K):
    def integ(phi, i):
        f = heston_cf(phi, S0, v0, r, kappa, theta, xi, rho, T)[i]
        return (np.exp(-1j*phi*np.log(K))*f/(1j*phi)).real
    P1 = 0.5 + quad(integ, 1e-8, 200, args=(0,), limit=200)[0]/np.pi
    P2 = 0.5 + quad(integ, 1e-8, 200, args=(1,), limit=200)[0]/np.pi
    return S0*P1 - K*np.exp(-r*T)*P2

### Construire les deux grilles de prix (tabulation)

On ne peut pas appeler `heston_call` (une integration de Fourier, ~4 ms) des millions de fois dans la boucle de couverture. On **tabule** donc le prix une fois sur une grille $(S, v, \tau)$ et on interpole ensuite (le notebook 11b montre pourquoi c'est precis). Il faut deux grilles, car les deux instruments n'ont pas la meme maturite :

- **passif** (call $K=100$, $T_1=1$) : sur l'horizon $t \in [0,1]$, la maturite residuelle $\tau = T_1 - t$ balaie $[0.01, 1]$ (bornee a 0.01, le pricer degenere en $\tau \to 0$) ;
- **couverture** (call $K_h=100$, $T_2=2$) : $\tau = T_2 - t$ balaie $[1, 2]$.

Les bornes en $S$ (55 a 175) et en $v$ (0.005 a 0.15) couvrent la ou les trajectoires Heston vont realistement, pour eviter d'extrapoler. Cette cellule coute environ **1 minute** (une seule fois).

In [ ]:
def build_grid(Sg, vg, tg, Kstrike):
    """Tabule le prix Heston C(S, v, tau) sur la grille, pour un strike donne."""
    grid = np.zeros((len(Sg), len(vg), len(tg)))
    for i, S in enumerate(Sg):                       # boucle sur le prix
        for j, v in enumerate(vg):                   # boucle sur la variance
            for k, tau in enumerate(tg):             # boucle sur la maturite residuelle
                grid[i, j, k] = heston_call(S, v, r, kappa, theta, xi, rho, tau, Kstrike)
    return grid

Sg  = np.linspace(55, 175, 25)      # axe prix     : 25 noeuds
vg  = np.linspace(0.005, 0.15, 12)  # axe variance : 12 noeuds
tgO = np.linspace(1.0, 2.0, 13)     # maturite residuelle de la couverture : [1, 2]
tgL = np.linspace(0.01, 1.0, 15)    # maturite residuelle du passif        : [0.01, 1]

t0 = time.time()
gridO = build_grid(Sg, vg, tgO, Kh)                  # grille option de couverture (~50 s)
gridL = build_grid(Sg, vg, tgL, K)                   # grille passif            (~20 s)
print(f"grilles construites en {time.time()-t0:.0f} s")

# interpolateurs trilineaires : (S, v, tau) -> prix
Ointerp = RegularGridInterpolator((Sg, vg, tgO), gridO, bounds_error=False, fill_value=None)
Pinterp = RegularGridInterpolator((Sg, vg, tgL), gridL, bounds_error=False, fill_value=None)

premium = float(Pinterp([[S0, v0, T1]])[0])          # prime encaissee en vendant le passif
print(f"prime du passif (prix Heston) = {premium:.3f}")

### Chaque ligne des trois blocs ci-dessus

- **Imports et params.** `quad` (integration) et `RegularGridInterpolator` (interpolation) de scipy ; `torch` pour le reseau. `theta=0.04` est la **variance** de long terme, soit une vol $\sqrt{0.04}=20\%$. `Kh, T2` : l'option de couverture est un call strike 100 de maturite **2 ans** ; a l'echeance du passif ($t=1$) il lui reste $\tau = T_2-T_1 = 1$ an, donc il vaut encore quelque chose et on peut le revendre.
- **Le pricer** `heston_cf` / `heston_call` : identique au notebook 11 (fonction caracteristique + inversion de Fourier, deux probabilites $P_1, P_2$, prix $= S_0 P_1 - K e^{-rT} P_2$).
- **`build_grid`** : la triple boucle remplit la case `grid[i,j,k]` avec le prix exact au point $(S_i, v_j, \tau_k)$. `enumerate` donne a la fois l'indice et la valeur.
- **Les axes** : `np.linspace(a, b, N)` cree $N$ valeurs regulierement espacees. Deux axes tau differents (`tgO`, `tgL`) car les deux options n'ont pas la meme maturite.
- **`Ointerp`, `Pinterp`** : a partir de la grille remplie, scipy construit l'interpolant **trilineaire**. Pour un point quelconque $(S,v,\tau)$, il prend les 8 sommets du petit cube qui l'entoure et fait une moyenne ponderee lineaire. `bounds_error=False, fill_value=None` = extrapolation lineaire si on sort de la grille.
- **`premium`** : le prix du passif en $t=0$, lu dans sa grille a $(S_0, v_0, \tau=T_1)$ ; c'est ce qu'on encaisse en vendant le call.

## 2. Pré-simulation : un paquet de trajectoires, puis les prix d'option dessus

On simule Heston par Euler *full-truncation* (variance tronquée à 0, comme aux notebooks précédents). On génère **deux** paquets : un pour l'entraînement (grand, ré-échantillonné en mini-batchs) et un pour le test **hors échantillon** (trajectoires jamais vues, pour un CVaR honnête). Puis, sur chaque paquet, on pré-calcule le long de la trajectoire :

- $O_k = C_{\text{Heston}}(S_k, v_k, T_2 - t_k)$ : le prix de l'option de couverture à chaque date (ce qu'on paie/reçoit en la tradant) ;
- $O_{\text{fin}} = C_{\text{Heston}}(S_{T_1}, v_{T_1}, T_2 - T_1)$ : sa valeur de revente à l'échéance du passif.


In [ ]:
def sim_heston(S0, v0, drift, T, n, m):
    """Euler full-truncation. Renvoie (S, v) de forme (m, n+1)."""
    dt = T/n
    S = np.empty((m, n+1)); v = np.empty((m, n+1)); S[:,0] = S0; v[:,0] = v0
    for k in range(n):
        Z1 = rng.standard_normal(m)
        Z2 = rho*Z1 + np.sqrt(1-rho**2)*rng.standard_normal(m)   # correlation instantanee rho
        vk = np.maximum(v[:,k], 0.0)                              # full-truncation : v- = max(v,0)
        v[:,k+1] = np.maximum(v[:,k] + kappa*(theta-vk)*dt + xi*np.sqrt(vk*dt)*Z2, 0.0)
        S[:,k+1] = S[:,k]*np.exp((drift - 0.5*vk)*dt + np.sqrt(vk*dt)*Z1)
    return S, v

def price_along(S, v, interp, Kstrike, Tmat):
    """Prix de l'option (maturite Tmat) le long de la trajectoire : tau = Tmat - t."""
    m, n1 = S.shape; times = np.linspace(0, T1, n1)
    O = np.empty_like(S)
    for k in range(n1):
        tau = max(Tmat - times[k], 1e-3)
        O[:,k] = interp(np.c_[S[:,k], v[:,k], np.full(m, tau)])
    return O

m_tr, m_te = 100_000, 80_000
S_tr, v_tr = sim_heston(S0, v0, mu, T1, n, m_tr)
S_te, v_te = sim_heston(S0, v0, mu, T1, n, m_te)
O_tr = price_along(S_tr, v_tr, Ointerp, Kh, T2)     # prix couverture le long des trajectoires (train)
O_te = price_along(S_te, v_te, Ointerp, Kh, T2)     # idem (test)
print("paquets simules :", S_tr.shape, S_te.shape)
print(f"O couverture en t0 = {O_tr[0,0]:.3f}   |   O a l'echeance (moyen) = {O_tr[:,-1].mean():.3f}")


### Pourquoi ces lignes

- `sim_heston` : la boucle séquentielle Heston. À chaque pas on tire $Z_1$ (choc du prix) et $Z_2$ corrélé à $Z_1$ par $\rho$ (choc de la variance). `vk = max(v,0)` puis re-`max(...,0)` sur $v_{k+1}$ : c'est le schéma *full-truncation*, qui empêche la variance de devenir négative (artefact d'Euler). Le prix utilise la forme exponentielle exacte du pas, avec la correction d'Itô $-\tfrac12 v_k\,dt$.
- `price_along` : pour chaque date $t_k$, on price l'option à la maturité résiduelle $\tau = T_{\text{mat}} - t_k$ via l'interpolateur. On empile $(S_k, v_k, \tau)$ et on appelle l'interpolateur en une fois (vectorisé sur les $m$ trajectoires).
- On génère **train** (100k) et **test** (80k), avec le *même* générateur mais des tirages différents : les trajectoires de test sont donc distinctes, ce qui donne un CVaR hors échantillon honnête.
- `O_tr[0,0]` doit valoir le prix Heston du call $(K_h=100, T_2=2)$ en $(S_0, v_0)$, soit environ 12.9 (call plus long, donc plus cher que le passif à 8.7).


## 3. Benchmark classique : la couverture delta-vega

La couverture classique neutralise **deux** grecques du portefeuille (short passif $P$, long $h_S$ sous-jacents et $h_O$ options de couverture $O$). On veut :

$$\underbrace{h_S + h_O\,\frac{\partial O}{\partial S}}_{\text{delta du portefeuille}} = \frac{\partial P}{\partial S}, \qquad \underbrace{h_O\,\frac{\partial O}{\partial v}}_{\text{vega du portefeuille}} = \frac{\partial P}{\partial v}.$$

(le sous-jacent a $\partial S/\partial v = 0$, il n'aide pas sur le vega ; d'où le $h_O$ pour le vega, puis $h_S$ pour le delta résiduel.) On résout de haut en bas :

$$h_O = \frac{\partial P/\partial v}{\partial O/\partial v}, \qquad h_S = \frac{\partial P}{\partial S} - h_O\,\frac{\partial O}{\partial S}.$$

Les grecques sont calculées par **différences finies** sur les interpolateurs Heston. On rééquilibre à chaque pas (couverture continue idéalisée), en payant les coûts sur les deux instruments.


In [ ]:
hS_, hv_ = 1.0, 0.005    # pas des differences finies (en prix, en variance)
def greeks(interp, Sk, vk, tau):
    """Renvoie (prix, dP/dS, dP/dv) par differences finies centrees."""
    tau = np.full_like(Sk, tau)
    Pp  = interp(np.c_[Sk+hS_, vk, tau]); Pm = interp(np.c_[Sk-hS_, vk, tau])
    vm  = np.maximum(vk-hv_, 1e-4)
    Pvp = interp(np.c_[Sk, vk+hv_, tau]); Pvm = interp(np.c_[Sk, vm, tau])
    dS  = (Pp - Pm)/(2*hS_)
    dv  = (Pvp - Pvm)/(vk+hv_ - vm)
    return dS, dv

def cvar(pnl, a=0.95):
    loss = -pnl
    return loss[loss >= np.quantile(loss, a)].mean()

times = np.linspace(0, T1, n+1)

def classic_hedge(S, v, O, vega=True):
    m = S.shape[0]; cash = np.full(m, premium); posS = np.zeros(m); posO = np.zeros(m)
    for k in range(n):
        tauL = max(T1 - times[k], 0.01); tauO = T2 - times[k]
        PS, Pv = greeks(Pinterp, S[:,k], v[:,k], tauL)
        if vega:
            OS, Ov = greeks(Ointerp, S[:,k], v[:,k], tauO)
            tgtO = Pv/Ov                      # vega-neutre
            tgtS = PS - tgtO*OS               # delta-neutre residuel
            trO = tgtO - posO
            cash -= trO*O[:,k] + cost*np.abs(trO)*O[:,k]   # cout sur l'option
            posO = tgtO
        else:
            tgtS = PS
        trS = tgtS - posS
        cash -= trS*S[:,k] + cost*np.abs(trS)*S[:,k]       # cout sur le sous-jacent
        posS = tgtS; cash *= np.exp(r*dt)
    return cash + posS*S[:,-1] + posO*O[:,-1] - np.maximum(S[:,-1]-K, 0.0)

pnl_delta = classic_hedge(S_te, v_te, O_te, vega=False)
pnl_dv    = classic_hedge(S_te, v_te, O_te, vega=True)
print(f"CVaR delta pur (grecque Heston) = {cvar(pnl_delta):.3f}")
print(f"CVaR delta-vega classique       = {cvar(pnl_dv):.3f}")


### Lecture du benchmark

- `greeks` : différences finies centrées. $\partial/\partial S \approx (P(S{+}h) - P(S{-}h))/2h$ ; pour $v$ on borne $v-h_v$ à $10^{-4}$ (variance positive) et on divise par le vrai écart utilisé au dénominateur.
- `classic_hedge` : la même mécanique de trésorerie que les notebooks 09/10, mais avec **deux** positions. Quand `vega=True`, on calcule d'abord $h_O = P_v/O_v$ (annule le vega), on trade l'option (avec son coût), puis $h_S = P_S - h_O O_S$ (annule le delta résiduel).
- À l'échéance : on paie le payoff du passif $(S_{T_1}-K)^+$, on revend les $h_S$ actions à $S_{T_1}$ et les $h_O$ options à $O_{\text{fin}} = O_{:,-1}$.
- Attendu : le delta pur plafonne autour de **8.8**, le delta-vega descend vers **6.5** : la preuve chiffrée que le vega était le morceau non couvert.


## 4. Le réseau multi-instruments

Maintenant le réseau. Il observe l'état et sort **deux** nombres : les positions cibles $(h_S, h_O)$. État en entrée (6 features) :

$$\big[\ \log(S/K),\ \ \tau=T_1-t,\ \ h_S^{\text{prec}},\ \ h_O^{\text{prec}},\ \ v,\ \ O/S_0\ \big].$$

On lui donne la position courante dans les deux instruments (pour qu'il « sente » les coûts de retrading) et le prix de l'option $O$ (pour qu'il sache ce que coûte un trade d'option). La perte est le **CVaR de Rockafellar-Uryasev** avec la variable auxiliaire $w$, exactement comme avant, mais le P&L intègre maintenant les deux instruments.


In [ ]:
def cvar_ru(loss, w, a=0.95):
    return w + torch.mean(torch.relu(loss - w))/(1.0 - a)

class HedgeNet2(torch.nn.Module):
    def __init__(self, h=48):
        super().__init__()
        self.net = torch.nn.Sequential(
            torch.nn.Linear(6, h), torch.nn.ReLU(),
            torch.nn.Linear(h, h), torch.nn.ReLU(),
            torch.nn.Linear(h, 2))          # 2 sorties : (position sous-jacent, position option)
    def forward(self, x):
        return self.net(x)

# tenseurs de donnees (marche exogene => constantes, aucun gradient a travers eux)
St  = torch.tensor(S_tr, dtype=torch.float32)
vt  = torch.tensor(v_tr, dtype=torch.float32)
Ot  = torch.tensor(O_tr, dtype=torch.float32)

def hedging_loss(net, w, idx):
    """P&L couvert sur le sous-paquet idx, puis CVaR."""
    S = St[idx]; v = vt[idx]; O = Ot[idx]; m = S.shape[0]
    cash = torch.full((m,), premium); posS = torch.zeros(m); posO = torch.zeros(m)
    for k in range(n):
        tau = float(T1 - k*dt)
        feat = torch.stack([torch.log(S[:,k]/K), torch.full((m,), tau),
                            posS, posO, v[:,k], O[:,k]/S0], dim=1)
        a = net(feat); aS = a[:,0]; aO = a[:,1]              # positions cibles
        trS = aS - posS; trO = aO - posO
        cash = cash - trS*S[:,k] - cost*torch.abs(trS)*S[:,k]   # trade + cout sous-jacent
        cash = cash - trO*O[:,k] - cost*torch.abs(trO)*O[:,k]   # trade + cout option
        cash = cash*np.exp(r*dt)
        posS = aS; posO = aO
    pnl = cash + posS*S[:,-1] + posO*O[:,-1] - torch.clamp(S[:,-1]-K, min=0.0)
    return cvar_ru(-pnl, w, alpha)


### Ce que fait chaque ligne

- `HedgeNet2` : un MLP $6 \to 48 \to 48 \to 2$. La dernière couche a **2 sorties** : c'est le seul vrai changement d'architecture par rapport au notebook 10. Pas d'activation en sortie : les positions peuvent être positives ou négatives, de n'importe quelle taille.
- `St, vt, Ot` : les trajectoires pré-simulées et les prix d'option, convertis en tenseurs **constants**. Ils ne portent pas de gradient : le marché est exogène. Le gradient du CVaR ne remonte qu'à travers `net(feat)`.
- Dans la boucle : `feat` empile les 6 features (dont les positions courantes `posS, posO` et le prix d'option normalisé `O/S0`). Le réseau sort `a` de forme $(m,2)$, on sépare `aS, aO`. On trade les deux instruments, on paie les deux coûts (proportionnels à $|{\rm trade}|\times{\rm prix}$), on capitalise la trésorerie.
- P&L final : trésorerie + valeur des deux positions revendues $-$ payoff du passif. On minimise le CVaR de la perte $-{\rm pnl}$.


In [ ]:
net = HedgeNet2()
w   = torch.zeros(1, requires_grad=True)                       # variable auxiliaire du CVaR
opt = torch.optim.Adam(list(net.parameters()) + [w], lr=1e-3)

batch = 20_000; epochs = 400
for ep in range(epochs):
    idx = torch.randint(0, m_tr, (batch,))                    # mini-batch tire dans le paquet train
    loss = hedging_loss(net, w, idx)
    opt.zero_grad(); loss.backward(); opt.step()
    if (ep+1) % 50 == 0:
        print(f"epoch {ep+1:4d}   CVaR train (batch) = {loss.item():.3f}")


On ré-échantillonne un mini-batch de 20 000 trajectoires à chaque époque dans le paquet d'entraînement : ça donne de la variété (le tail du CVaR change de batch en batch) sans re-simuler Heston. `w` est optimisé conjointement avec le réseau : à l'optimum $w \approx \mathrm{VaR}_\alpha$, et la valeur de la perte est le CVaR.


## 5. Évaluation hors échantillon et comparaison

On évalue les trois stratégies sur le **paquet de test** (jamais vu à l'entraînement) : delta pur, delta-vega classique, et le réseau multi-instruments.


In [ ]:
Ste = torch.tensor(S_te, dtype=torch.float32)
vte = torch.tensor(v_te, dtype=torch.float32)
Ote = torch.tensor(O_te, dtype=torch.float32)

def rollout_net(net):
    m = m_te; cash = torch.full((m,), premium); posS = torch.zeros(m); posO = torch.zeros(m)
    for k in range(n):
        tau = float(T1 - k*dt)
        feat = torch.stack([torch.log(Ste[:,k]/K), torch.full((m,), tau),
                            posS, posO, vte[:,k], Ote[:,k]/S0], dim=1)
        a = net(feat); aS = a[:,0]; aO = a[:,1]
        trS = aS - posS; trO = aO - posO
        cash = cash - trS*Ste[:,k] - cost*torch.abs(trS)*Ste[:,k]
        cash = cash - trO*Ote[:,k] - cost*torch.abs(trO)*Ote[:,k]
        cash = cash*np.exp(r*dt); posS = aS; posO = aO
    return (cash + posS*Ste[:,-1] + posO*Ote[:,-1] - torch.clamp(Ste[:,-1]-K, min=0.0)).detach().numpy()

net.eval()
with torch.no_grad():
    pnl_net = rollout_net(net)

print(f"CVaR delta pur          = {cvar(pnl_delta):.3f}")
print(f"CVaR delta-vega class.  = {cvar(pnl_dv):.3f}")
print(f"CVaR reseau (2 instr.)  = {cvar(pnl_net):.3f}")
print(f"rappel reseau 1 instr. (nb 10) ~ 6.90")


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))

for pnl, lab, c in [(pnl_delta,'delta pur','tab:red'),
                    (pnl_dv,'delta-vega classique','tab:orange'),
                    (pnl_net,'reseau 2 instruments','tab:green')]:
    ax1.hist(pnl, bins=140, range=(-30,10), histtype='step', lw=1.8, label=f"{lab} (CVaR {cvar(pnl):.2f})", color=c)
ax1.axvline(0, color='k', lw=0.6); ax1.set_title("Distribution du P&L couvert (test)")
ax1.set_xlabel("P&L"); ax1.legend(fontsize=8)

labels = ['delta\npur','delta-vega\nclassique','reseau\n2 instr.']
vals = [cvar(pnl_delta), cvar(pnl_dv), cvar(pnl_net)]
ax2.bar(labels, vals, color=['tab:red','tab:orange','tab:green'])
for i,val in enumerate(vals): ax2.text(i, val+0.05, f"{val:.2f}", ha='center')
ax2.set_title("CVaR 95% (plus bas = mieux)"); ax2.set_ylabel("CVaR")
fig.tight_layout(); fig.savefig("12_multi_instrument.png", dpi=110)
print("figure : 12_multi_instrument.png")


## Ce qu'il faut retenir (talk track d'entretien)

- **Pourquoi une deuxième option ?** Sous Heston le marché est incomplet : le vega (risque de variance) n'est pas couvrable avec le sous-jacent seul. On ajoute un call de maturité plus longue, qui porte du vega, comme second instrument.
- **L'astuce de calcul.** Marché exogène $\Rightarrow$ $(S_t, v_t)$ indépendants du réseau $\Rightarrow$ le prix de l'option de couverture est une *donnée* le long de la trajectoire. On pré-tabule le pricer Heston (fonction caractéristique, notebook 11) sur une grille $(S, v, \tau)$ et on interpole. Le gradient ne passe que par les actions.
- **Le réseau vs le classique.** Le delta-vega classique rééquilibre les deux grecques en continu et paie beaucoup de coûts ; le réseau optimise directement le CVaR *sous coûts*, donc il apprend à ne trader l'option que quand ça vaut la friction. C'est là qu'il gagne.
- **La morale du projet.** GBM sans coût : le réseau égale le delta (rien à gagner). Dès qu'on ajoute frictions et incomplétude, il bat les heuristiques classiques, parce qu'il optimise le vrai objectif de risque, pas une approximation grecque par grecque.
